# Route extraction from AIS data

TREAD-style pipeline: waypoints with DBSCAN, routes between waypoints, KDE of a route.
The logic lives in `ais_anomaly.routes`; this notebook runs it on the NOAA Hawaii data and plots the results.

Data: run `scripts/download_noaa.py` first (see README). Slow steps are cached in `data/cache/`: delete those files after changing the data or the parameters.

Figures and maps are also saved to `outputs/` (PNG for matplotlib, HTML for interactive maps).

## **Import libraries**



In [ ]:
from pathlib import Path

import folium
import hvplot.pandas  # noqa: F401  (registers DataFrame.hvplot)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colors as mcolors
from tqdm import tqdm

from ais_anomaly import routes
from ais_anomaly.maps import add_trips, waypoint_map
from ais_anomaly.io import cached, read_noaa, save_figure

DATA = Path("../data")
CACHE = DATA / "cache"

## **Load the dataset**



**National Ocenanic and Atmosphere Administration AIS dataset**
* **Location**:      Hawaii
* **Time period**:   2019-12-31 23:57:30 from  2020-03-30 23:58:54

Vessel traffic data or Automatic Identification Systems (AIS) are a navigation safety device that transmits and monitors the location and characteristics of many vessels in real-time.


The original records were filtered to a one-minute frequency rate and were subsetted to depict the location and description of vessels broadcasting within the Exclusive Economic Zone

Source -> https://marinecadastre.gov/accessais/

The dataset  contains AIS (Automatic Identification System) data related to a vessel. The features are:
* **MMSI**: Unique identifier for the ship (Maritime Mobile Service Identity).
*  **BaseDateTime**: Date and time of the AIS report.
* **LAT**: Latitude of the ship's position.
* **LON**: Longitude of the ship's position.
* **SOG**: Speed Over Ground, the speed of the ship relative to the ground.
* **COG**: Course Over Ground, the angle relative to the geographic north followed by the ship.
* **Heading**: The direction the ship is facing, in degrees.
* **VesselName**: Name of the ship.
* **IMO**: International Maritime Organization, a unique identifier assigned to ships.
* **CallSign**: Call sign of the ship.
* **VesselType**: Type of the ship.
* **Status**: The ship's status (might indicate whether it is anchored, in motion, etc.) very often the fields are null.
* **Length**: Length of the ship.
* **Width**: Width of the ship.
* **Draft**: Draft of the ship.
* **Cargo**: Type of cargo being carried.
* **TransceiverClass**: Class of the ship's transceiver.

In [ ]:
# January 2020, created by: python scripts/download_noaa.py --start 2020-01-01 --end 2020-01-31 --region hawaii_2020Q1 --out hawaii_2020_01
# (for the whole quarter: --end 2020-03-30 and read "hawaii_2020Q1.parquet")
AIS_FILE = DATA / "hawaii_2020_01.parquet"
ais_data = read_noaa(AIS_FILE)
print(f"Dataframe size: {ais_data.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
ais_data.head()

In [ ]:
print(f"Rows: {ais_data.shape[0]}  Columns: {ais_data.shape[1]}")
print(f"Time period {ais_data['BaseDateTime'].iloc[0]} to {ais_data['BaseDateTime'].iloc[-1]}")

Plot  some vessels positions using Latitude and Longitude information

In [ ]:
# Map centred on Hawaii, one marker every 10000 reports
m = folium.Map(location=[21, -158], tiles="OpenStreetMap", zoom_start=9)
for _, row in ais_data.iloc[::10000].iterrows():
    folium.Marker(location=[row["LAT"], row["LON"]]).add_to(m)
save_figure(m, "02_ais_overview_map")
m

## **Create a dictionary of dataframes, one for each ship identifier**


From the **AIS_data** dataframe, extract a list of AIS data related a **specific** ship.

In [ ]:
# One DataFrame per vessel (MMSI), keeping vessels with at least 50 reports
Vs = routes.split_by_vessel(ais_data, min_points=50)
print(f"Number of vessels with at least 50 reports: {len(Vs)}")

## **Estimate the average speed of each vessel**

Compute the estimate of the average speed of each vessel, based on LAT and LON

Update the status based on estimated speed because COG is not reliable (as AIA partner told us)

In [ ]:
# avg_speed from consecutive LAT/LON (haversine), status = stationary | sailing
Vs = [routes.add_speed_status(v) for v in tqdm(Vs)]

## **Dataframe Waypoints**

Compute the WayPoints from AIS data


Waypoint Objects (WPs): Clustering discontinuous events initiated by different vessel objects results in the formation of waypoint objects (WPs). These WPs identify stationary points (POs), entry points (ENs), and exit points (EXs) within the selected bounding box

* POs -> stationary points
* ENs -> entry points 
* EXs -> exit points

WP dataframe is union of POs ENs EXs

In [ ]:
WP = routes.waypoint_candidates(Vs)
print(f"Waypoint candidates: {len(WP)}")


### **Use DBSCAN clustering to identify the WayPoints**
DBSCAN stands for *Density-Based Spatial Clustering of Applications with Noise*, is a clustering algorithm used in machine learning and data analysis.  Unlike traditional clustering algorithms such as K-means, DBSCAN does not require specifying the number of clusters beforehand. Instead, it identifies clusters based on the density of data points in the features space.

**WARNING: this requires a lot of RAM memory**


In [ ]:
# eps and min_samples are tuned by hand: the goal is to find clusters in port areas
labels, WP_dataframe_hull = cached(
    CACHE / "waypoints.pkl", lambda: routes.cluster_waypoints(WP, eps=0.2, min_samples=20)
)
subsetWP = WP[["LAT", "LON"]].assign(LABEL=labels)
clusters_plot = subsetWP.hvplot.scatter(x="LON", y="LAT", by="LABEL")
save_figure(clusters_plot, "02_dbscan_clusters")
clusters_plot

Evaluate the DBSCAN results

In [ ]:
print(f"Number of clusters (noise excluded): {len(WP_dataframe_hull)}")
plt.figure(figsize=(10, 10))
plt.plot(subsetWP["LON"], subsetWP["LAT"], "o")
for hull in WP_dataframe_hull["hull"]:
    if hull.geom_type == "Polygon":  # degenerate clusters have a line/point hull
        plt.plot(*hull.exterior.xy, "g", lw=5)
save_figure(plt.gcf(), "02_waypoint_hulls")

Create a dataframe with the convex hulls of the identified clusters and plot on a geographic map

In [ ]:
m = waypoint_map(WP_dataframe_hull, zoom=8)
folium.LatLngPopup().add_to(m)
save_figure(m, "02_waypoints_map")
m

## **Verify if AIS points for each vessel are inside the specified waypoints**




In [ ]:
def assign_all():
    for v in tqdm(Vs):
        v["waypoints"] = routes.assign_waypoints(v, WP_dataframe_hull)
    return Vs


Vs = cached(CACHE / "vessels_with_waypoints.pkl", assign_all)
pd.concat(Vs)["waypoints"].value_counts()

## **Extracting possible routes**


* I am interested in vessels that pass through more than 2 waypoints.
* From the vessel dataframe, check whether there are vessels with a number of different elements in the 'waypoints' column equal to 3 (considering '-1').
* This means that a vessel to have traveled at least one route between two distinct waypoints

 **possible_vessel_route dictonary a subset of dictornary Vs that have completed at least one route**


In [ ]:
possible_vessel_route = [v for v in Vs if v["waypoints"].nunique() >= 3]
print(f"Vessels with at least one route: {len(possible_vessel_route)}")
possible_vessel_route[0].head()

In [ ]:
Route_object = pd.concat([r.assign(trip=i) for i, r in enumerate(r for v in possible_vessel_route for r in routes.extract_routes(v))])

Route object is always an dataframe divided by vessels but with the waypoints and Route fields inside

In [ ]:
print(Route_object.groupby("Route").size())

#### This graph displays the routes taken by a single ship.
* Time instances are represented with a color scale from the most distant instant (Blue) to the most recent (Red).
* It is noted that the ship has a gap in the AIS data from 26/2/2020 03:30 to 16/03/2020 22:06, as it appears in two different locations.

In [ ]:
subset_1 = Route_object[Route_object["VesselName"] == "LADY BETTY"].sort_values("BaseDateTime")

# one colour per trip, from blue (oldest trip) to red (newest)
trip_order = subset_1.groupby("trip")["BaseDateTime"].min().sort_values().index
cmap = mcolors.LinearSegmentedColormap.from_list("time", ["blue", "red"], N=max(len(trip_order), 2))
trip_colors = {trip: mcolors.to_hex(cmap(i)) for i, trip in enumerate(trip_order)}

m = waypoint_map(WP_dataframe_hull, zoom=4)
add_trips(m, subset_1, color=trip_colors)
save_figure(m, "02_route_lady_betty")
m

#### This graph displays the routes routes from WP3 waypoints to WP10 and WP10 to WP3 traveled by different ships

In [ ]:
m = waypoint_map(WP_dataframe_hull, zoom=4)
for route, color in [("WP_3-WP_10", "green"), ("WP_10-WP_3", "red")]:
    add_trips(m, Route_object[Route_object["Route"] == route], color=color, popup_col="VesselName")
save_figure(m, "02_routes_WP_3_WP_10")
m

## Anomaly route detection with **Kernel Density Estimation**


* Kernel Density Estimation (KDE) can be useful in identifying anomalous maritime traffic routes by providing a smooth estimate of the probability density function (PDF) for the geographical distribution of vessel positions.

* Kernel Density Estimation (KDE) is a non-parametric statistical technique used for estimating the probability density function (PDF) of a random variable. It provides a smooth estimate of the underlying probability distribution of a dataset. The basic idea behind KDE is to place a kernel (a smooth, symmetric function) at each data point and then sum up these kernels to create a continuous probability density function.

In [ ]:
subset_1 = Route_object[Route_object["Route"] == "WP_10-WP_3"]
X, Y, Z = routes.route_kde(subset_1["LON"], subset_1["LAT"])
extent = [X.min(), X.max(), Y.min(), Y.max()]

fig, ax = plt.subplots(figsize=(15, 15))
for hull in WP_dataframe_hull["hull"]:
    if hull.geom_type == "Polygon":
        ax.plot(*hull.exterior.xy, "b", lw=5)
ax.imshow(np.rot90(Z), cmap=plt.cm.gist_earth_r, extent=extent)
ax.grid(color="k", linestyle="-", linewidth=2, alpha=0.1)
ax.plot(subset_1["LON"], subset_1["LAT"], "k.", markersize=1)
cset = ax.contourf(X, Y, Z, cmap="Reds")
fig.colorbar(cset, shrink=0.6)
ax.set_xlim(extent[:2])
ax.set_ylim(extent[2:])
save_figure(fig, "02_kde_WP_10-WP_3")
plt.show()